# Kratos MCP Server — interactive sessions: watching a flow reach steady state

The lid-driven cavity (Re = 100, monolithic Navier–Stokes) marches in pseudo-time to a steady vortex. As a batch job you only see
the end state. With an **interactive session** we advance it in chunks and watch the solution settle: the velocity at a probe node,
the peak velocity and its location, and the pressure range — after every chunk — and stop when it has stopped changing.

Uses `session_start`, `session_advance`, `session_field_summary`, `session_inspect_node`, `session_model_info` and `session_close`
against the shipped `kratos://examples/lid-driven-cavity` case. Needs FluidDynamicsApplication and LinearSolversApplication. Every output is from a live run.

In [1]:
import asyncio
import base64
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from contextlib import AsyncExitStack

from IPython.display import Image as IPImage, Markdown, display

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def _preview(value, limit=4000):
    text = value if isinstance(value, str) else json.dumps(value, indent=2, default=str)
    if len(text) > limit:
        text = text[: limit] + f"\n... [truncated, {len(text)} chars total]"
    print(text)


async def call(tool_name, **kwargs):
    # Call an MCP tool, print its JSON result, display any inline images,
    # and return the parsed value for use in later cells.
    result = await session.call_tool(tool_name, kwargs)
    value = None
    for block in result.content:
        if block.type == "text":
            try:
                value = json.loads(block.text)
            except json.JSONDecodeError:
                value = block.text
        elif block.type == "image":
            display(IPImage(data=base64.b64decode(block.data), format=block.mimeType.split("/")[-1]))
    _preview(value)
    return value


async def read_resource(uri):
    result = await session.read_resource(uri)
    return result.contents[0].text


async def get_prompt(name, **kwargs):
    result = await session.get_prompt(name, kwargs)
    return result.messages[0].content.text

## Connect to the server

The server's stdio pipes are managed by anyio context managers whose cancel scopes are tied to the asyncio Task that opened them. Every Jupyter cell runs its top-level `await` in a *new* Task, so we run the whole connect → disconnect lifecycle inside one persistent background task kept alive on an `asyncio.Event`, and only *signal* it to close. We start this demo server with `KRATOS_MCP_ALLOW_EXEC=1` so the opt-in `session_exec_python` tool works.

In [2]:
session = None
_stop_event = asyncio.Event()
_ready_event = asyncio.Event()


async def _server_lifecycle():
    global session
    async with AsyncExitStack() as stack:
        # session_exec_python is opt-in: enable it for this demo server only
        params = StdioServerParameters(
            command=sys.executable, args=["-m", "kratos_mcp.server"],
            env={**os.environ, "KRATOS_MCP_ALLOW_EXEC": "1"})
        read, write = await stack.enter_async_context(stdio_client(params))
        session = await stack.enter_async_context(ClientSession(read, write))
        await session.initialize()
        _ready_event.set()
        await _stop_event.wait()  # keeps this task (and the session) alive


_server_task = asyncio.create_task(_server_lifecycle())
await _ready_event.wait()

tools = await session.list_tools()
resources = await session.list_resources()
prompts = await session.list_prompts()
print(f"{len(tools.tools)} tools, {len(resources.resources)} resources, "
      f"{len(prompts.prompts)} prompts")
print("session tools:", sorted(t.name for t in tools.tools if t.name.startswith("session_")))
print("prompts:", [p.name for p in prompts.prompts])

64 tools, 14 resources, 6 prompts
session tools: ['session_advance', 'session_close', 'session_exec_python', 'session_field_summary', 'session_get_parameters', 'session_inspect_element', 'session_inspect_node', 'session_list', 'session_model_info', 'session_set_nodal_values', 'session_start', 'session_status']
prompts: ['setup_structural_analysis', 'setup_thermal_analysis', 'setup_fluid_analysis', 'debug_failed_simulation', 'postprocess_results', 'interactive_session']


## 1. Stage the case

The documented result of this example is a minimum centerline velocity of ≈ −0.116 m/s near y = 0.40 (the recirculation). On the 11 × 11 node
grid that point is node **50** (x = 0.5, y = 0.4) — our probe.

In [3]:
print((await read_resource("kratos://examples/lid-driven-cavity")).split("## Verified result")[1][:900])



Running this exact case (30 pseudo-time steps to a steady state, ~1 s
wall-clock) and reading the final VTK snapshot:

    lid velocity u on the top row .................. 1.00 m/s (imposed)
    vertical centerline u(y) at x = 0.5, minimum ... -0.116 m/s at y = 0.40

The negative interior velocity is the signature of the recirculation: fluid
dragged forward along the lid returns backward through the cavity interior,
so u(y) reverses sign below the lid. On a fine (Ghia) mesh at Re = 100 that
minimum is about -0.21 at y ~= 0.46; this coarse 10x10 mesh under-resolves the
vortex core (expected -- refine with more divisions for a sharper profile).

## Reproducing this with the tools

```
mdpa_create_structured_mesh(path='case/mesh.mdpa', kind='rectangle',
                             size=[1.0, 1.0], divisions=[10, 10],
                             element_name='Element2D3N', triangles=true


In [4]:
import kratos_mcp
workdir = Path(tempfile.mkdtemp(prefix="kratos-mcp-session-cavity-"))
example_dir = Path(kratos_mcp.__file__).parent / "examples" / "lid_driven_cavity"
for f in ("mesh.mdpa", "ProjectParameters.json", "Materials.json"):
    shutil.copy(example_dir / f, workdir / f)
print("case directory:", workdir)
await call("validate_case", case_dir=str(workdir))

case directory: /tmp/kratos-mcp-session-cavity-skhhgtfh


{
  "valid": true,
  "issues": [],
  "warnings": [],
  "deep_validated": true
}


{'valid': True, 'issues': [], 'warnings': [], 'deep_validated': True}

## 2. Open the session

In [5]:
start = await call("session_start", case_dir=str(workdir), isolate=True)
sid = start["session_id"]

{
  "session_id": "20261010-110524-db0c05",
  "state": "ready",
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110524-db0c05/execution",
  "log_file": "/root/.kratos-mcp/sessions/20261010-110524-db0c05/session.log",
  "analysis_class": "KratosMultiphysics.FluidDynamicsApplication.fluid_dynamics_analysis:FluidDynamicsAnalysis",
  "time": 0.0,
  "end_time": 30.0,
  "step": 0,
  "steps_taken": 0,
  "finished": false,
  "finalized": false,
  "model_part": "FluidModelPart",
  "nodes": 121,
  "elements": 200,
  "conditions": 40,
  "model_parts": [
    "FluidModelPart",
    "FluidModelPart.fluid_computational_model_part",
    "FluidModelPart.corner",
    "FluidModelPart.bottom",
    "FluidModelPart.top",
    "FluidModelPart.left",
    "FluidModelPart.right",
    "FluidModelPart.domain"
  ]
}


In [6]:
await call("session_get_parameters", session_id=sid, path="/solver_settings/time_stepping")
await call("session_model_info", session_id=sid, depth=1)

{
  "path": "/solver_settings/time_stepping",
  "parameters": {
    "automatic_time_step": false,
    "time_step": 1.0
  }
}
{
  "name": "FluidModelPart",
  "nodes": 121,
  "elements": 200,
  "conditions": 40,
  "sub_model_parts": [
    {
      "name": "fluid_computational_model_part",
      "nodes": 121,
      "elements": 200,
      "conditions": 40
    },
    {
      "name": "corner",
      "nodes": 1,
      "elements": 0,
      "conditions": 0
    },
    {
      "name": "bottom",
      "nodes": 11,
      "elements": 0,
      "conditions": 10
    },
    {
      "name": "top",
      "nodes": 11,
      "elements": 0,
      "conditions": 10
    },
    {
      "name": "left",
      "nodes": 11,
      "elements": 0,
      "conditions": 10
    },
    {
      "name": "right",
      "nodes": 11,
      "elements": 0,
      "conditions": 10
    },
    {
      "name": "domain",
      "nodes": 121,
      "elements": 200,
      "conditions": 0
    }
  ],
  "process_info": {
    "TIME": 0.0,
    "

{'name': 'FluidModelPart',
 'nodes': 121,
 'elements': 200,
 'conditions': 40,
 'sub_model_parts': [{'name': 'fluid_computational_model_part',
   'nodes': 121,
   'elements': 200,
   'conditions': 40},
  {'name': 'corner', 'nodes': 1, 'elements': 0, 'conditions': 0},
  {'name': 'bottom', 'nodes': 11, 'elements': 0, 'conditions': 10},
  {'name': 'top', 'nodes': 11, 'elements': 0, 'conditions': 10},
  {'name': 'left', 'nodes': 11, 'elements': 0, 'conditions': 10},
  {'name': 'right', 'nodes': 11, 'elements': 0, 'conditions': 10},
  {'name': 'domain', 'nodes': 121, 'elements': 200, 'conditions': 0}],
 'process_info': {'TIME': 0.0, 'STEP': 0, 'DELTA_TIME': 1.0, 'DOMAIN_SIZE': 2},
 'buffer_size': 2,
 'available_model_parts': ['FluidModelPart',
  'FluidModelPart.fluid_computational_model_part',
  'FluidModelPart.corner',
  'FluidModelPart.bottom',
  'FluidModelPart.top',
  'FluidModelPart.left',
  'FluidModelPart.right',
  'FluidModelPart.domain']}

## 3. March in chunks of five steps

After each chunk we record the probe velocity, the maximum velocity magnitude (and where it occurs) and the pressure range, plus whether
every step's nonlinear solve converged. The probe value should settle onto the documented ≈ −0.116 m/s.

In [7]:
async def snapshot(call_name, **kw):
    result = await session.call_tool(call_name, kw)
    return json.loads(result.content[0].text)

history = []
while True:
    adv = await snapshot("session_advance", session_id=sid, steps=5)
    probe = await snapshot("session_inspect_node", session_id=sid, node_id=50, variables=["VELOCITY", "PRESSURE"])
    vel = await snapshot("session_field_summary", session_id=sid, variable="VELOCITY")
    prs = await snapshot("session_field_summary", session_id=sid, variable="PRESSURE")
    history.append(dict(t=adv["time"], u=probe["values"]["VELOCITY"][0], umax=vel["max"],
                        where=vel["argmax"]["coordinates"][:2], pmin=prs["component_min"][0] if "component_min" in prs else prs["min"],
                        pmax=prs["max"], conv=all(h["converged"] for h in adv["history"])))
    if adv["finished"]:
        break

print(f"{'t':>5} {'u(0.5,0.4)':>11} {'max |u|':>8} {'at (x, y)':>14} {'p max':>9} {'converged':>10}")
for h in history:
    print(f"{h['t']:5.1f} {h['u']:11.4f} {h['umax']:8.4f} {str(tuple(round(c, 2) for c in h['where'])):>14} {h['pmax']:9.4f} {str(h['conv']):>10}")

    t  u(0.5,0.4)  max |u|      at (x, y)     p max  converged
  5.0     -0.1080   1.0000     (0.0, 1.0)   22.8296       True
 10.0     -0.1152   1.0000     (0.0, 1.0)   22.9739       True
 15.0     -0.1155   1.0000     (0.0, 1.0)   22.9823       True
 20.0     -0.1155   1.0000     (0.0, 1.0)   22.9827       True
 25.0     -0.1155   1.0000     (0.0, 1.0)   22.9827       True
 30.0     -0.1155   1.0000     (0.0, 1.0)   22.9827       True


## 4. Has it reached steady state?

Compare the probe velocity between the last two chunks, and with the documented value.

In [8]:
last, prev = history[-1]["u"], history[-2]["u"]
print(f"probe u at t={history[-2]['t']:.0f}: {prev:.5f}")
print(f"probe u at t={history[-1]['t']:.0f}: {last:.5f}   (change {abs(last - prev) / abs(last):.2%})")
print(f"documented minimum centerline u: -0.116")

probe u at t=25: -0.11552
probe u at t=30: -0.11552   (change 0.00%)
documented minimum centerline u: -0.116


The lid imposes u = 1 on the top row, so `max |u|` is pinned at 1.0 on the boundary from the start (the first step already imposes it);
the *interior* probe is what evolves. Let's read the final state with the top row excluded by sampling a few interior nodes:

In [9]:
for node_id in (50, 61, 72):   # (0.5, 0.4), (0.5, 0.5), (0.5, 0.6) on the vertical centerline
    n = await snapshot("session_inspect_node", session_id=sid, node_id=node_id, variables=["VELOCITY", "PRESSURE"])
    print(node_id, [round(c, 3) for c in n["coordinates"][:2]], "u =", round(n["values"]["VELOCITY"][0], 4),
          " v =", round(n["values"]["VELOCITY"][1], 4), " p =", round(n["values"]["PRESSURE"], 4))

50 [0.5, 0.4] u = -0.1155  v = 0.0172  p = -0.4566
61 [0.5, 0.5] u = -0.1117  v = 0.0345  p = -1.208
72 [0.5, 0.6] u = -0.0755  v = 0.0535  p = -2.1278


## 5. Close the session

In [10]:
await call("session_close", session_id=sid)
await call("session_list")

{
  "session_id": "20261010-110524-db0c05",
  "state": "closed",
  "finalized": true
}
{
  "session_id": "20261010-110502-379dbd",
  "state": "closed",
  "case_dir": "/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution",
  "analysis_class": "KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis",
  "age_seconds": 23.4
}


{'session_id': '20261010-110502-379dbd',
 'state': 'closed',
 'case_dir': '/root/.kratos-mcp/sessions/20261010-110502-379dbd/execution',
 'analysis_class': 'KratosMultiphysics.StructuralMechanicsApplication.structural_mechanics_analysis:StructuralMechanicsAnalysis',
 'age_seconds': 23.4}

## Cleanup

Close the client session (which stops the server subprocess, and with it any session still open).

In [11]:
_stop_event.set()      # wake the lifecycle task up ...
await _server_task     # ... and let it exit its own AsyncExitStack from the task that opened it
print("session closed, server subprocess terminated")
print("case directory left on disk:", workdir)

session closed, server subprocess terminated
case directory left on disk: /tmp/kratos-mcp-session-cavity-skhhgtfh
